# Vancouver Business Licences, 1997 to 2026: Building the Dataset

Built for a group project in BAIT 518 Data Visualization (UBC Sauder School of Business, Master of Business Analytics), October 2026. See the README for the project summary.

This notebook combines the City of Vancouver's three business licence datasets into two clean files for Tableau:

| Output | One row per | Use it for |
|---|---|---|
| `businesses_by_year.csv` | business location per year | anything over time: openings vs. closures, employees by year |
| `businesses_summary.csv` | business location | lifespan, survival, closures, size, plus an employees column for each year |

Each step below starts with the **decision** we made, **why**, and the **trade-off**, followed by a quick check of what the step did to the data.

**Inputs** (semicolon-separated CSVs from [opendata.vancouver.ca](https://opendata.vancouver.ca)), saved in the `data/` folder (see the README for download links):
- `business-licences-1997-to-2012.csv`
- `business-licences-2013-to-2024.csv`
- `business-licences.csv` (current years)
- `local-area-boundary.geojson` (optional; the "Local area boundary" dataset exported as GeoJSON)

## 0. Settings
Raw files are read from `data/` and results are written to `output/`. Run the notebook from the repository's top folder, or change `DATA_DIR` below.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

FILES = {
    "1997-2012": "business-licences-1997-to-2012.csv",
    "2013-2024": "business-licences-2013-to-2024.csv",
    "current":   "business-licences.csv",
}
BOUNDARY_FILE = "local-area-boundary.geojson"

CHAIN_MIN_LOCATIONS = 5   # a name at this many addresses or more counts as a chain
INCLUDE_YEAR_COLUMNS = False   # True adds employees_1997...2026 and licensed_1997...2026 to the summary file

## 1. Load the three files

**Decision:** read every column as text, and only the columns we need.

**Why:**
- The City's export uses semicolons, so `sep=";"` is required. With the default comma, any business name containing a comma breaks the row.
- The three files store some columns differently. For example, `LocalArea` is a number in the older files and text in the current one. Reading everything as text and converting each column ourselves keeps the three files consistent.
- Keeping 15 of the 25 columns cuts memory use roughly in half on ~1.9 million rows.

In [ ]:
USE_COLS = [
    "FOLDERYEAR", "LicenceRSN", "BusinessName", "BusinessTradeName", "Status",
    "IssuedDate", "BusinessType", "BusinessSubType", "Unit", "House", "Street",
    "PostalCode", "LocalArea", "NumberofEmployees", "Geom",
]

frames = []
for label, fname in FILES.items():
    df = pd.read_csv(DATA_DIR / fname, sep=";", dtype=str, low_memory=False,
                     usecols=lambda c: c in USE_COLS)
    df["source"] = label
    frames.append(df)
    print(f"{fname}: {len(df):,} rows")

raw = pd.concat(frames, ignore_index=True)
print(f"\nTotal: {len(raw):,} licence records")

## 2. Fix the year and remove the overlap between files

**Decision 1:** convert `FOLDERYEAR` from two digits to four. 97 becomes 1997, 5 becomes 2005, 24 becomes 2024.

**Decision 2:** drop the 11 records from 1996. Our analysis starts in 1997, when the City's records begin in full.

**Decision 3:** 2024 appears in both the 2013 to 2024 file and the current file. We remove duplicates using `LicenceRSN` (the City's unique licence ID) and keep the copy from the **newest** file, because it carries the most recent status (for example, a licence later marked "Gone Out of Business").

In [ ]:
fy = pd.to_numeric(raw["FOLDERYEAR"], errors="coerce")
raw["year"] = np.where(fy >= 90, 1900 + fy, 2000 + fy).astype(int)
raw = raw[raw["year"] >= 1997]

before = len(raw)
newest_first = {"current": 0, "2013-2024": 1, "1997-2012": 2}
raw = (raw.assign(_order=raw["source"].map(newest_first))
          .sort_values("_order")
          .drop_duplicates(subset="LicenceRSN", keep="first")
          .drop(columns="_order"))
print(f"Removed {before - len(raw):,} duplicate licences. {len(raw):,} remain.")
print(f"Years: {raw['year'].min()} to {raw['year'].max()}")

## 3. Remove applications that never became licences

**Decision:** drop records with status **Pending** (applications still in review) and the single **Invalid Status Code** record.

**Why:** a pending application doesn't mean the business ever operated. Keeping these would inflate openings.

The statuses we keep: **Issued**, **Gone Out of Business**, **Inactive** and **Cancelled**.

In [ ]:
print(raw["Status"].value_counts().to_string(), "\n")
raw = raw[~raw["Status"].isin(["Pending", "Invalid Status Code"])]
print(f"{len(raw):,} records after removing pending applications")

## 4. Group ~250 business types into 16 general categories

**The problem:** in 2024 the City replaced its whole list of business types. The old types are now marked `*Historic*`, and the new ones have different names (for example, "Restaurant Class 1 *Historic*" became "Restaurant"). Comparing industries across 30 years means putting old and new types into the same groups.

**Decision:** map every old and new type to one of 16 broad categories. Judgment calls worth knowing about:
- **Gas stations** are counted as **Retail**.
- **Printing** is counted as **Manufacturing**, and **photography** as **Personal Services**.
- **Fitness centres** are counted as **Arts, Entertainment and Recreation**.
- **Homecraft** (small home-based production) is counted as **Personal Services**.

**Residential Rental:** licences for rental housing (houses, apartments, secondary suites, short-term rentals) are kept, but with their own category and a true/false column, `is_residential_rental`. That way Tableau can show all-licence totals, or filter rentals out to look at businesses only.

**Administrative (removed):** one-off permits that aren't ongoing operations: temporary filming permits, liquor licence applications and amendments, sign permits, and 2010 Winter Games licences. Counting these would inflate openings and closures.

Any type not in the list becomes **Other** and is saved to `unmapped_business_types.csv` so we can check it.

In [ ]:
CATEGORIES = {
    "Food and Drink": [
        "Restaurant Class 1", "Restaurant Class 2", "Restaurant", "Restaurant Class 1 & Karaoke",
        "Ltd Service Food Establishment", "Limited Service Food Establishment",
        "Dining Lounge", "Dining Lounge/Room", "Caterer", "Lounge 'A'", "Cabaret",
        "Public House", "Neighbourhood Pub", "Hotel Lounge", "Club Lounge", "Marine Pub",
        "Liquor Establishment Standard", "Liquor Establishment Extended", "Liquor Establishment",
        "U-Brew/U-Vin", "Street Vendor", "Peddler - Food", "CNIB Concession - Food",
        "Food Market", "Public Market Operator-Annual", "Public Market - Daily",
    ],
    "Retail": [
        "Retail Dealer", "Retail Dealer - Food", "Retail Dealer - Grocery",
        "Retail Dealer - Market Outlet", "Retail Dealer - Used Goods", "Retail Dealer - Cannabis",
        "Grocery Store", "Liquor Retail Store", "Specialty Wine Store", "Secondhand Dealer",
        "Retail with Anc. Secondhand", "Jeweller", "Pet Store", "Pawnbroker",
        "Adult Entertainment Store", "Adult Retail Store", "Auto Dealer", "Machinery Dealer",
        "Lumber Yard", "Christmas Tree Lot", "Pharmacy", "Junk Dealer", "Peddler",
        "Transient Trader/Peddler-A", "Transient Trader/Peddler-W", "Auctioneer",
        "Vending Machines", "Gasoline Station", "Self-serve Station", "Conventional Gas Station",
        "Split-island Station", "Card Lock Station", "Gas Station - Marine",
        "Gas Station - Propane", "Gas Station", "Marine Service Station", "Oil Gas and Other Fuels",
    ],
    "Wholesale and Manufacturing": [
        "Wholesale Dealer", "Wholesale Dealer - Food", "Wholesale Dealer - Non-Food",
        "Wholesale Dealer w/ Anc. Retail", "Wholesale Dealer - Food with Anc. Retail",
        "Manufacturer", "Manufacturer - Food", "Manufacturer with Anc. Retail",
        "Manufacturer - Food with Anc. Retail", "Food Processing", "Product Assembly",
        "Sheet Metal Works", "Non-Food Manufacturer Assembler and Processor",
        "Food Manufacturer Assembler and Processor", "Creative Products Manufacturer",
        "Warehouse Operator - Non-Food", "Warehouse Operator - Food", "Auto Wholesaler",
        "Printing Services", "Blueprint Printing",
    ],
    "Construction and Trades": [
        "Contractor", "Contractor - Special Trades", "General Contractor", "Trade Contractor",
        "Electrical Contractor", "Electrical-Security Alarm Installation", "Plumber",
        "Plumber & Gas Contractor", "Plumber Sprinkler & Gas Contractor",
        "Plumber & Sprinkler Contractor", "Gas Contractor", "Sprinkler Contractor",
        "Sprinkler & Gas Contractor", "Roofer", "Painter", "Landscape Gardener",
        "Equipment Operator", "Building Repair and Maintenance",
    ],
    "Professional and Office Services": [
        "Office", "Computer Services", "Consulting and Management Services", "Legal Services",
        "Architectural and Engineering Services", "Design Services",
        "Information Communication Technology", "Business Support Services", "Business Services",
        "Employment Agency", "Referral Services",
        "Marketing Public Relations Advertising and Event Promotion Services",
        "Publishing and Journalism Services", "Talent Agency", "Booking Agency", "Model Agency",
        "Artist Agency", "Specialized Services", "Mining Services", "Forestry Services",
        "Telecommunications", "Laboratory Services",
    ],
    "Finance and Real Estate": [
        "Financial Services", "Financial Institution", "Money Services", "Cheque Cashing",
        "Insurance Services", "Brokerage Services", "Real Estate Dealer",
        "Real Estate Services", "Property Management",
    ],
    "Health and Wellness": [
        "Health Services", "Health Care Professionals and Services", "Health Care Facility",
        "Massage Therapist", "Physical Therapist", "Acupuncturist", "Herbalist",
        "Therapeutic Touch Technique", "Private Hospital", "Personal Care Home",
        "Health Enhancement Services", "Steam Bath",
    ],
    "Personal and Beauty Services": [
        "Hairdressing Salon", "Hairdresser", "Beauty Services", "Health and Beauty",
        "Cosmetologist", "Esthetician", "Special Beauty Culturist", "Nail Technician",
        "Electrolysis", "Tanning Salon", "Tattoo Parlour", "Personal Services", "Dry Cleaner",
        "Laundry-Coin Operated Services", "Laundry Depot", "Laundry (w/equipment)",
        "Laundry Services", "Seamstress/Tailor", "Boot & Shoe Repairs", "Dating Services",
        "Psychic/Fortune Teller", "Social Escort Services", "Body Rub Parlour", "Adult Services",
        "Funeral Services", "Animal Services", "Animal Clinic/Hospital", "Veterinarian",
        "Photographer", "Photo Services", "Printing Imaging and Photo Services", "Homecraft",
    ],
    "Repair, Auto and Maintenance": [
        "Repair/ Service/Maintenance", "General Repair and Maintenance", "Auto Repairs",
        "Auto Painter & Body Shop", "Auto Detailing", "Auto Washer", "Auto Wrecker",
        "Vehicle Repair Detailing and Washing Services", "Janitorial Services",
        "Carpet/Upholstery Cleaner", "Window Cleaner", "Power/ Pressure Washing",
        "Pest Control/Exterminator", "Chimney Sweep", "Locksmith", "Piano Tuner",
        "Security Services", "Marine Services",
    ],
    "Arts, Entertainment and Recreation": [
        "Entertainment Services", "Production Company", "Studio",
        "Photography Production and Rehearsal Studio", "Artist", "Artist Studio",
        "Artist Live/Work Studio", "Theatre", "Venue", "Exhibitions/Shows/Concerts", "Club",
        "Dance Hall", "Late Night Dance Event", "Billiard Room Keeper", "Arcade", "Bowling Alley",
        "Casino", "Bingo Hall", "Bingo Hall / Casino / Horse Racing", "Horse Racing",
        "Amusement Park", "Entertainment Centre", "Entertainment Facility",
        "Family Sports & Entertain Ctr", "Stadium/Concert Hall", "Hall / Spectator Sports Venue",
        "Assembly Hall", "Exhibition Centre", "Exotic Dancers",
        "Digital Entertainment and Interactive Technology", "Special Events",
        "Fitness Centre", "Sport and Fitness Instruction", "Livery & Feed Stables",
    ],
    "Education": [
        "Instruction", "Educational", "School (Business & Trade)", "Private School or College",
        "ESL Instruction", "Business - Vocational Instruction", "Arts and Creative Instruction",
    ],
    "Accommodation and Tourism": [
        "Hotel or Motel", "Motel", "Bed and Breakfast", "Tourism Services", "Travel Agent",
        "Boat Charter Services", "Marina Operator", "Trailer Court",
    ],
    "Transportation and Logistics": [
        "Moving/Transfer Service", "Logistics Services", "Transportation and Support Services",
        "Auto Parking Lot/Parkade", "Parking Area / Garage", "Backyard Pay Parking",
        "Postal Rental Agency", "Scavenging", "Waste Collection and Hauling Services",
        "Recycling Depot", "Recycling and Resource Recovery Services", "Bottle Depot",
        "Dairy Delivery Services", "Liquor Delivery Services", "Rentals", "Rental Services",
    ],
    "Community and Non-profit": [
        "Community Association", "Community Association Manager", "Association or Society",
        "Soliciting For Charity",
    ],
    "Agriculture": ["Agriculture", "Urban Farm Class A", "Urban Farm Class B"],
    "Residential Rental": [
        "Single Detached House", "Apartment House", "Apartment House Strata",
        "Apartment House-99 Year Lease", "Multiple Dwelling", "Duplex",
        "Secondary Suite - Permanent", "Secondary Suite - Phase Out", "Pre-1956 Dwelling",
        "Residential/Commercial", "Long-term Rental", "Short-term Rental Operator",
        "Non-profit Housing", "Live-aboards", "Rooming House",
    ],
    "Administrative (removed)": [
        "Temporary Filming Company", "Temp Liquor Licence Amendment", "Liquor License Application",
        "Cannabis Licence Application", "Sign Permit", "Provincial Liquor Review",
        "2010 Winter Games", "Club Manager",
    ],
}

# Remove "*Historic*" and extra spaces so old and new names can be matched
raw["business_type"] = (raw["BusinessType"].fillna("")
                        .str.replace("*Historic*", "", regex=False)
                        .str.replace(r"\s+", " ", regex=True).str.strip())

type_to_category = {t: cat for cat, types in CATEGORIES.items() for t in types}
raw["category"] = raw["business_type"].map(type_to_category).fillna("Other")

unmapped = (raw.loc[raw["category"] == "Other", "business_type"].value_counts()
               .rename_axis("business_type").reset_index(name="records"))
unmapped.to_csv(OUTPUT_DIR / "unmapped_business_types.csv", index=False)
print(f"Unmapped business types: {len(unmapped)} (saved to unmapped_business_types.csv)")

raw = raw[raw["category"] != "Administrative (removed)"]
raw["is_residential_rental"] = raw["category"] == "Residential Rental"
print(f"{len(raw):,} records after removing administrative permits\n")
print(raw["category"].value_counts().to_string())

## 5. Decide what counts as "one business"

**The problem:** there's no business ID that stays the same across years. Licence numbers start with the year (`97-123456`), so every renewal gets a new number.

**Decision:** one business = **one legal business name at one address**.

**Why:**
- **Legal name rather than trade name:** the trade name ("Pottery Barn #936") is blank about 60% of the time, while the legal name ("Williams-Sonoma Canada Inc") is almost always filled in. We still *show* the trade name when there is one, since it's what customers recognize.
- **Address included:** a chain like Starbucks has a separate licence, staff and employee count at each location, and each location can open or close on its own. So 30 Starbucks addresses = 30 businesses.

**Standardizing text so the same business matches across years:**
- Names: uppercase, remove punctuation, and remove legal endings (Ltd, Inc, Corp), so "Joe's Cafe Ltd." and "JOES CAFE LTD" match.
- Addresses: uppercase and shorten street words, so "Fraser Street" and "FRASER ST" match.

**Trade-offs:**
- A business that **moves** or **changes its legal name** will look like one business closing and a new one opening.
- About half of licences have **no address** (home-based businesses, and some types the City withholds for privacy). These are matched on name only.

In [ ]:
LEGAL_ENDINGS = r"\b(LTD|LIMITED|INC|INCORPORATED|CORP|CORPORATION|CO|COMPANY|LLP|LLC|ULC)\b"
STREET_WORDS = {
    "STREET": "ST", "AVENUE": "AV", "AVE": "AV", "DRIVE": "DR", "ROAD": "RD",
    "BOULEVARD": "BLVD", "PLACE": "PL", "CRESCENT": "CRES", "COURT": "CT",
    "HIGHWAY": "HWY", "LANE": "LN", "SQUARE": "SQ", "WEST": "W", "EAST": "E",
    "NORTH": "N", "SOUTH": "S", "SUITE": "", "UNIT": "",
}

def standardize_name(s):
    s = s.fillna("").astype(str).str.upper().str.replace(r"[^A-Z0-9 ]", " ", regex=True)
    s = s.str.replace(LEGAL_ENDINGS, " ", regex=True)
    return s.str.replace(r"\s+", " ", regex=True).str.strip()

def standardize_address(s):
    s = s.fillna("").astype(str).str.upper().str.replace(r"[^A-Z0-9 ]", " ", regex=True)
    for word, short in STREET_WORDS.items():
        s = s.str.replace(rf"\b{word}\b", short, regex=True)
    return s.str.replace(r"\s+", " ", regex=True).str.strip()

legal = raw["BusinessName"].fillna("").str.strip()
trade = raw["BusinessTradeName"].fillna("").str.strip()
raw["legal_name"] = legal
raw["business_name"] = trade.where(trade != "", legal)            # what we display
raw["name_key"] = standardize_name(legal.where(legal != "", trade))  # what we match on

raw["address"] = (raw["Unit"].fillna("") + " " + raw["House"].fillna("") + " "
                  + raw["Street"].fillna("")).str.replace(r"\s+", " ", regex=True).str.strip()
raw["address_key"] = standardize_address(raw["address"])
raw.loc[raw["address_key"] == "", "address_key"] = "NO ADDRESS"

raw = raw[raw["name_key"] != ""]
raw["business_id"] = raw.groupby(["name_key", "address_key"], sort=False).ngroup() + 1

print(f"{raw['business_id'].nunique():,} unique businesses from {len(raw):,} licences")
print(f"Licences with no address: {(raw['address_key'] == 'NO ADDRESS').mean():.0%}")

## 6. Clean employees, dates and coordinates

**Employees.** **Decision:** treat **0 as unknown**, not as "no employees."
**Why:** the City's documentation says 0 can mean either. Before 2013, 92% of licences show 0, which isn't believable as "no staff." It reflects that the field mostly wasn't collected.
**Trade-off:** employee analysis is only reliable from about **2013 onward**, and sole proprietors who truly have no staff are counted as unknown.

**Dates.** The files store dates in two formats (`2006-02-03` and `2020-02-28T14:34:52-08:00`). Both are converted to a plain date.

**Coordinates.** The `Geom` column holds each location as text like `{"coordinates": [-123.1, 49.2]}`. We split it into separate `lat` and `lon` number columns, which Tableau can map directly.

In [ ]:
emp = pd.to_numeric(raw["NumberofEmployees"], errors="coerce")
raw["employees"] = emp.where(emp > 0)

raw["issued_date"] = (pd.to_datetime(raw["IssuedDate"], errors="coerce", utc=True, format="ISO8601")
                        .dt.tz_localize(None).dt.normalize())

def split_coordinates(geom):
    lon = np.full(len(geom), np.nan)
    lat = np.full(len(geom), np.nan)
    for i, g in enumerate(geom.values):
        if isinstance(g, str) and "coordinates" in g:
            try:
                c = json.loads(g)["coordinates"]
                lon[i], lat[i] = float(c[0]), float(c[1])
            except (ValueError, KeyError, IndexError, TypeError):
                pass
    return lat, lon

raw["lat"], raw["lon"] = split_coordinates(raw["Geom"])
raw["LocalArea"] = raw["LocalArea"].where(~raw["LocalArea"].isin(["", "nan"]))

by_year_known = raw.assign(known=raw["employees"].notna()).groupby("year")["known"].mean()
print("Share of licences with a known employee count, selected years:")
print(by_year_known.loc[lambda s: s.index.isin([1997, 2005, 2012, 2013, 2020, 2025])].map("{:.0%}".format).to_string())
print(f"\nLicences with coordinates: {raw['lat'].notna().mean():.0%}")

## 7. One row per business per year

**The problem:** some businesses hold **more than one licence in the same year** at the same address, for example a replacement licence, or a separate licence for a second activity such as a restaurant that also sells liquor.

**Decisions:**
- **Employees:** take the **highest** count across that year's licences, **never the sum**. Summing would double count the same staff.
- **Status:** keep the **most serious** status that year (Gone Out of Business, then Cancelled, then Inactive, then Issued). If any licence that year says the business closed, the year is marked as closed.
- **Everything else** (name, type, address): use the latest licence that year.

In [ ]:
severity = {"Issued": 1, "Inactive": 2, "Cancelled": 3, "Gone Out of Business": 4}
raw["status_rank"] = raw["Status"].map(severity).fillna(0)
raw = raw.sort_values(["business_id", "year", "status_rank"])

by_year = raw.groupby(["business_id", "year"], as_index=False).agg(
    business_name=("business_name", "last"),
    legal_name=("legal_name", "last"),
    address=("address", "last"),
    postal_code=("PostalCode", "last"),
    business_type=("business_type", "last"),
    category=("category", "last"),
    is_residential_rental=("is_residential_rental", "last"),
    status=("Status", "last"),               # sorted by severity, so "last" = most serious
    employees=("employees", "max"),          # max, not sum: no double counting
    local_area=("LocalArea", "last"),
    lat=("lat", "last"),
    lon=("lon", "last"),
    licences_that_year=("LicenceRSN", "nunique"),
    first_issued=("issued_date", "min"),
)
multi = (by_year["licences_that_year"] > 1).sum()
print(f"{len(by_year):,} business-years. {multi:,} of them had more than one licence and were merged.")

## 8. Fill in neighbourhoods for years before 2024

**The problem:** `LocalArea` (one of the City's 22 neighbourhoods) is **blank for every record before 2024**. Only the current file has it.

**Decision:** fill it in two ways, in this order:
1. **Same business, later year:** if a business has a neighbourhood in its 2024 to 2026 licences, use it for all its earlier years at the same address.
2. **Map lookup:** for businesses that closed before 2024, place their coordinates on the City's official neighbourhood boundaries (`local-area-boundary.geojson`) and read off which area they fall in. This step needs the `geopandas` library (`pip install geopandas`) and is skipped if it or the file is missing.

**Trade-off:** businesses with no address have no coordinates, so they stay **"Unknown."** Neighbourhood charts therefore show businesses with a storefront or office, not home-based ones. "Out of Town" means the business is licensed in Vancouver but based elsewhere.

In [ ]:
known_area = by_year.dropna(subset=["local_area"]).groupby("business_id")["local_area"].last()
by_year["local_area"] = by_year["local_area"].fillna(by_year["business_id"].map(known_area))
print(f"After step 1 (same business): {by_year['local_area'].notna().mean():.0%} have a neighbourhood")

boundary_path = DATA_DIR / BOUNDARY_FILE
try:
    import geopandas as gpd
    if boundary_path.exists():
        areas = gpd.read_file(boundary_path)
        cols = {c.lower(): c for c in areas.columns}
        name_col = cols.get("name") or cols.get("local_area") or cols.get("mapid")
        need = by_year["local_area"].isna() & by_year["lat"].notna()
        points = gpd.GeoDataFrame(
            index=by_year.index[need],
            geometry=gpd.points_from_xy(by_year.loc[need, "lon"], by_year.loc[need, "lat"]),
            crs="EPSG:4326").to_crs(areas.crs)
        joined = gpd.sjoin(points, areas[[name_col, "geometry"]], how="left", predicate="within")
        joined = joined[~joined.index.duplicated()]
        by_year.loc[joined.index, "local_area"] = joined[name_col]
        print(f"After step 2 (map lookup): {by_year['local_area'].notna().mean():.0%} have a neighbourhood")
    else:
        print(f"{BOUNDARY_FILE} not found, skipping the map lookup")
except ImportError:
    print("geopandas not installed, skipping the map lookup")

by_year["local_area"] = by_year["local_area"].fillna("Unknown")

## 9. Flag chains

**Decision:** a business name that appears at **5 or more different addresses** (over all 30 years) is flagged as a chain.

**Why:** it lets us compare chains with independent businesses, for example whether chain locations last longer, which matters to someone opening an independent shop. The cut-off of 5 is our choice: it's high enough to leave out small local businesses with two or three locations. Change `CHAIN_MIN_LOCATIONS` in the settings to try other cut-offs.

In [ ]:
ids_and_names = raw[["business_id", "name_key"]].drop_duplicates("business_id")
locations_per_name = ids_and_names.groupby("name_key")["business_id"].nunique()
chain_names = set(locations_per_name[locations_per_name >= CHAIN_MIN_LOCATIONS].index)
by_year["is_chain"] = by_year["business_id"].map(
    ids_and_names.set_index("business_id")["name_key"]).isin(chain_names)
print(f"{len(chain_names):,} chain names, covering {by_year.drop_duplicates('business_id')['is_chain'].sum():,} locations")

## 10. One row per business: lifespan, closure and survival

**How each measure is defined:**

| Column | Definition | Why |
|---|---|---|
| `first_year`, `last_year` | first and last year the business held a licence | |
| `closed` | closed = no licence in the most recent year (2026) | businesses renew every January, so by late 2026 a business with no 2026 licence has very likely stopped |
| `lifespan_years` | last year minus first year, plus 1 | a business licensed only in 2010 has a lifespan of 1 year |
| `opened_before_data` | first seen in 1997 | it may have opened years earlier; we can't see its real start. Used to build the next column; not saved |
| `complete_lifespan` | closed **and** opened after 1997 | **use this filter for lifespan charts**, so the averages only include businesses whose start and end we both see |
| `survived_5_years` | 1 if still licensed 5 years after opening, 0 if not, blank if it opened too recently to know | stored as 1/0 so that **averaging it in Tableau gives the survival rate** directly |
| `size_group` | 1-4, 5-19, 20-99, 100+ employees (highest count ever reported) | "Unknown" if it never reported a count |
| `opening_period` | opened 1998 to 2009, 2010 to 2021, 2022 or later, or open before the data starts | for "then vs. now" comparisons. 2021 is the last year with a full 5 years to judge survival, so "2022 or later" has no survival rate yet |
| `closed_in_2024_transition` | last licensed year was 2024 | 2024 closures are inflated by the City's licence system change (see step 13), so this lets us flag or exclude them |
| `employees_YYYY`, `licensed_YYYY` | employees and licensed yes/no (1/0) for every year. **Only added when `INCLUDE_YEAR_COLUMNS = True`** | off by default: the yearly file already holds this, one row per year, which is the layout Tableau uses. These 60 mostly blank columns are only handy for reading one business's history across a row in Excel |

**Known simplification:** if a business skips a year and comes back, we treat it as continuous. `years_licensed` shows how many years it actually held a licence, so gaps can be spotted.

In [ ]:
data_start, data_end = int(by_year["year"].min()), int(by_year["year"].max())

summary = (by_year.sort_values("year").groupby("business_id").agg(
    business_name=("business_name", "last"),
    legal_name=("legal_name", "last"),
    address=("address", "last"),
    postal_code=("postal_code", "last"),
    business_type=("business_type", "last"),
    category=("category", "last"),
    is_residential_rental=("is_residential_rental", "last"),
    local_area=("local_area", "last"),
    lat=("lat", "last"),
    lon=("lon", "last"),
    is_chain=("is_chain", "last"),
    first_year=("year", "min"),
    last_year=("year", "max"),
    years_licensed=("year", "nunique"),
    ever_gone_out_of_business=("status", lambda s: (s == "Gone Out of Business").any()),
    max_employees=("employees", "max"),
    median_employees=("employees", "median"),
).reset_index())

summary["still_open"] = summary["last_year"] == data_end
summary["closed"] = ~summary["still_open"]
summary["closure_year"] = summary["last_year"].where(summary["closed"]).astype("Int64")
summary["lifespan_years"] = summary["last_year"] - summary["first_year"] + 1
summary["opened_before_data"] = summary["first_year"] == data_start
summary["complete_lifespan"] = summary["closed"] & ~summary["opened_before_data"]

can_judge = summary["first_year"] + 5 <= data_end
summary["survived_5_years"] = (summary["lifespan_years"] > 5).astype("Int64").where(can_judge)

summary["size_group"] = pd.cut(summary["max_employees"], [0, 4, 19, 99, np.inf],
                               labels=["1-4", "5-19", "20-99", "100+"]).astype(str)
summary.loc[summary["max_employees"].isna(), "size_group"] = "Unknown"

summary["opening_period"] = np.select(
    [summary["opened_before_data"], summary["first_year"] <= 2009, summary["first_year"] <= 2021],
    ["Open before 1998 (start unknown)", "Opened 1998 to 2009", "Opened 2010 to 2021"],
    default="Opened 2022 or later")
summary["closed_in_2024_transition"] = summary["closure_year"].eq(2024).fillna(False).astype(bool)

if INCLUDE_YEAR_COLUMNS:
    employees_wide = by_year.pivot(index="business_id", columns="year", values="employees")
    employees_wide.columns = [f"employees_{c}" for c in employees_wide.columns]
    licensed_wide = by_year.assign(v=1).pivot(index="business_id", columns="year", values="v")
    licensed_wide = licensed_wide.fillna(0).astype(int)
    licensed_wide.columns = [f"licensed_{c}" for c in licensed_wide.columns]
    summary = summary.join(employees_wide, on="business_id").join(licensed_wide, on="business_id")

print(f"{len(summary):,} businesses, {data_start} to {data_end}")
print(f"Still open: {summary['still_open'].mean():.0%}")
print(f"Complete lifespans available: {summary['complete_lifespan'].sum():,}")
print(f"5-year survival (businesses only): "
      f"{summary.loc[~summary['is_residential_rental'], 'survived_5_years'].mean():.0%}")

## 11. Opening and closing flags for the yearly file

**Decision:**
- `is_opening_year` = the business's first year, **except 1997**. Businesses already open when the records start aren't new openings.
- `is_closing_year` = the business's last year, **except 2026**. Businesses licensed this year haven't closed.

In Tableau, **SUM(is_opening_year)** and **SUM(is_closing_year)** by year give the openings vs. closures chart.

In [ ]:
by_year = by_year.merge(summary[["business_id", "first_year", "last_year", "size_group"]],
                        on="business_id")
by_year["is_opening_year"] = ((by_year["year"] == by_year["first_year"])
                              & (by_year["year"] > data_start)).astype(int)
by_year["is_closing_year"] = ((by_year["year"] == by_year["last_year"])
                              & (by_year["year"] < data_end)).astype(int)

check = (by_year[~by_year["is_residential_rental"]]
         .groupby("year")[["is_opening_year", "is_closing_year"]].sum()
         .rename(columns={"is_opening_year": "openings", "is_closing_year": "closures"}))
print("Openings and closures, businesses only (last 8 years):")
print(check.tail(8).to_string())

## 12. Save for Tableau

**Decision:** save only the columns the charts need, and store each fact once.

**Why:** the yearly file has 1.5 million rows, so every column in it is repeated for every year a business exists. Details that never change from year to year (name, address, coordinates, type, size group) live once in the summary file. In Tableau, the two files are linked on `business_id`, so a yearly chart can still use any summary column.

| File | Columns saved |
|---|---|
| `businesses_by_year.csv` | `business_id`, `year`, `status`, `employees`, `is_opening_year`, `is_closing_year`, plus the three most-used filters: `category`, `local_area`, `is_residential_rental` |
| `businesses_summary.csv` | everything about the business: name, address, type, category, neighbourhood, coordinates, chain flag, first and last year, lifespan, closure, survival, size, opening period, 2024 flag |

Columns used only for checking (legal name, postal code, licences merged per year, median employees and similar) stay in memory for step 13 but aren't saved.

**Trade-off:** the yearly file on its own shows IDs, not names. **Always link it to the summary file on `business_id` in Tableau.**

In [ ]:
BY_YEAR_COLUMNS = ["business_id", "year", "category", "is_residential_rental", "local_area",
                   "status", "employees", "is_opening_year", "is_closing_year"]
SUMMARY_COLUMNS = ["business_id", "business_name", "address", "business_type", "category",
                   "is_residential_rental", "local_area", "lat", "lon", "is_chain",
                   "first_year", "last_year", "years_licensed", "max_employees", "closed",
                   "closure_year", "lifespan_years", "complete_lifespan", "survived_5_years",
                   "size_group", "opening_period", "closed_in_2024_transition"]
SUMMARY_COLUMNS += [c for c in summary.columns if c.startswith(("employees_", "licensed_"))]

by_year[BY_YEAR_COLUMNS].to_csv(OUTPUT_DIR / "businesses_by_year.csv", index=False, encoding="utf-8")
summary[SUMMARY_COLUMNS].to_csv(OUTPUT_DIR / "businesses_summary.csv", index=False, encoding="utf-8")
print(f"Saved businesses_by_year.csv: {len(by_year):,} rows, {len(BY_YEAR_COLUMNS)} columns")
print(f"Saved businesses_summary.csv: {len(summary):,} rows, {len(SUMMARY_COLUMNS)} columns")

## 13. Quality checks

These checks don't change the data. They test whether the results can be trusted, and are what we used to find the issues described in the README. Rerun them whenever the data or a decision changes.

**Check A: neighbourhood coverage.** Coverage should be close to the share of records with coordinates, since businesses with no address can never be placed. Any neighbourhood name in our data that isn't in the boundary file won't appear on the Tableau map.

In [ ]:
print(f"Business-years with a neighbourhood: {(by_year['local_area'] != 'Unknown').mean():.0%}")
print(f"Business-years with coordinates:     {by_year['lat'].notna().mean():.0%}")

if boundary_path.exists():
    features = json.load(open(boundary_path, encoding="utf-8"))["features"]
    def area_name(props):
        lower = {k.lower(): v for k, v in props.items()}
        return lower.get("name") or lower.get("local_area") or lower.get("mapid")
    map_names = {area_name(f["properties"]) for f in features}
    ours = set(by_year["local_area"]) - {"Unknown", "Out of Town"}
    print("Names in our data but not on the map:", sorted(ours - map_names) or "none")

**Check B: is the 2024 closure spike real?** 2024 is when the City replaced its licence types and system. If our matching broke at that point, the same businesses would "close" in 2024 and "reopen" in 2025 at the same address or under the same name. We compare that rate with normal years.

**Last result:** 16% at the same address and 3% under the same name in 2024, in line with 16 to 19% and 4 to 9% in 2018 to 2023. The matching held up.

In [ ]:
keys = raw.drop_duplicates("business_id").set_index("business_id")[["name_key", "address_key"]]
s = summary.join(keys, on="business_id")
s = s[~s["is_residential_rental"]]

def reappeared(close_year):
    closed = s[s["last_year"] == close_year]
    opened = s[s["first_year"] == close_year + 1]
    has_addr = closed["address_key"] != "NO ADDRESS"
    same_addr = has_addr & closed["address_key"].isin(set(opened["address_key"]))
    same_name = closed["name_key"].isin(set(opened["name_key"]))
    print(f"Closed {close_year}: {len(closed):,} | same address opened {close_year + 1}: {same_addr.mean():.0%}"
          f" | same name opened {close_year + 1}: {same_name.mean():.0%}")

for y in [2018, 2019, 2022, 2023, 2024]:
    reappeared(y)

**Check C: what kind of businesses make up the 2024 spike?** If the extra closures sit in one industry, a rule change is the likelier cause; if they're spread out, they're more likely real.

**Last result:** Construction and Trades went from 843 closures in 2023 to 2,091 in 2024, about 1,250 of the ~3,000 extra. That matches contractors moving to the Inter-Municipal Business Licence (not confirmed). The rest is spread across categories and looks like real closures. Use `closed_in_2024_transition` to flag these.

In [ ]:
s2 = s.merge(by_year[["business_id", "year", "status"]],
             left_on=["business_id", "last_year"], right_on=["business_id", "year"])

def profile(year):
    c = s2[s2["last_year"] == year]
    return pd.DataFrame({
        "closures": [len(c)],
        "status = Gone Out of Business": [f"{(c['status'] == 'Gone Out of Business').mean():.0%}"],
        "status = Issued (just not renewed)": [f"{(c['status'] == 'Issued').mean():.0%}"],
        "no address": [f"{(c['address_key'] == 'NO ADDRESS').mean():.0%}"],
        "only licensed 1 year": [f"{(c['lifespan_years'] == 1).mean():.0%}"],
    }, index=[year])

print(pd.concat([profile(y) for y in [2019, 2022, 2023, 2024]]).T.to_string(), "\n")

by_cat = pd.crosstab(s["category"], s["last_year"])[[2022, 2023, 2024]]
by_cat["2024 vs 2023"] = by_cat[2024] - by_cat[2023]
print(by_cat.sort_values("2024 vs 2023", ascending=False).to_string())

**Check D: are the "chains" real chains?** Lists the 20 names with the most locations.

**Last result:** parking operators, the big banks, Starbucks, H&R Block and Chevron, so the flag works. Parking lots dominate, so leave Transportation and Logistics out of any chain vs. independent comparison.

In [ ]:
print(s[s["is_chain"]].groupby("name_key").size().sort_values(ascending=False).head(20).to_string())